# 02 — Causal Analysis Walkthrough
**Congestion Pricing Impact Analyzer**

This notebook walks through all five causal layers in sequence, explains the
motivation for each, and interprets results.

| Layer | Method | Question |
|---|---|---|
| L1 | Naive TWFE DiD | What is the average treatment effect? |
| L2 | Event Study + CS-DiD | Did effects emerge gradually? Pre-trends? |
| L3 | Continuous Treatment DiD | What is the dose-response curve? |
| L4 | Synthetic DiD | Can we construct a better counterfactual? |
| L5a | Double ML | Does the ATE hold under ML-controlled confounding? |
| L5b | Causal Forest | Which zones / times were most affected? |

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path

PROC    = Path('../artifacts/processed')
RESULTS = Path('../artifacts/results')
FIGS    = Path('../artifacts/figures')
SHOCK   = pd.Timestamp('2025-01-05')

# Load enriched panel
pf = PROC / 'zone_day_panel_features.parquet'
if not pf.exists():
    pf = PROC / 'zone_day_panel_enriched.parquet'
panel = pd.read_parquet(pf)
panel['date'] = pd.to_datetime(panel['date'])

# Load feature cols
import pickle
fc_path = RESULTS / 'feature_cols.pkl'
feature_cols = pickle.load(open(fc_path, 'rb')) if fc_path.exists() else []

print(f'Panel: {panel.shape}  |  Features: {len(feature_cols)}')

## L1 — Naive Two-Way Fixed Effects DiD

**Model:** $Y_{it} = \alpha_i + \gamma_t + \beta(\text{Treat}_i \times \text{Post}_t) + \varepsilon_{it}$

**Limitation:** Assumes parallel trends unconditionally and ignores heterogeneous effects across zones.
We include this as a baseline and will show how subsequent estimators refine it.

In [ ]:
from src.causal.run_did import run_naive_did

outcomes = ['log_trip_count', 'avg_distance', 'avg_duration', 'avg_fare']
outcomes = [o for o in outcomes if o in panel.columns]

l1_results = {}
for outcome in outcomes:
    l1_results[outcome] = run_naive_did(panel, outcome=outcome)

# Display
summary = pd.DataFrame([
    {'Outcome': k, 'ATT': v['ate'], 'SE': v['se'],
     'p': v['p_value'], '95% CI': f"[{v['ci_low']:.4f}, {v['ci_high']:.4f}]"}
    for k, v in l1_results.items()
])
summary['Sig'] = summary['p'].apply(
    lambda p: '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else '')))
summary['Pct Change'] = summary.apply(
    lambda r: f"{(np.exp(r['ATT'])-1)*100:+.1f}%" if 'log' in r['Outcome'] else '—', axis=1)
print(summary.to_string(index=False))

## L2 — Event Study: Testing Parallel Trends and Dynamic Effects

The event study plots $\beta_k$ for each relative week $k$:
- **Pre-period ($k < 0$):** should be ≈ 0 (parallel trends test)
- **Post-period ($k \geq 0$):** dynamic treatment effects

In [ ]:
from src.causal.run_did import run_event_study
from src.evaluation.plots import plot_event_study

es_df = run_event_study(panel, outcome='log_trip_count', window=16)

fig = plot_event_study(es_df, outcome_label='log(Trip Count)')
plt.show()

pre_trend_p = es_df.attrs.get('pre_trend_pvalue', None)
print(f'\nPre-trend joint test p = {pre_trend_p:.4f}')
print('→ Parallel trends assumption', '✓ holds' if pre_trend_p > 0.05 else '⚠ may be violated')

## L2b — Callaway-Sant'Anna DiD

More honest ATT(g,t) estimates with valid bootstrap standard errors.
Avoids negative-weighting bias from staggered treatment.

In [ ]:
from src.causal.run_did import run_callaway_santanna
from src.evaluation.plots import plot_cs_did

cs = run_callaway_santanna(panel, outcome='log_trip_count', n_bootstrap=199)

print(f"CS-DiD Aggregated ATT = {cs['agg_att']:.4f}")
print(f"Pre-trend p           = {cs['pre_trend_pvalue']:.4f}")

fig = plot_cs_did(cs['att_gt'])
plt.show()

## L3 — Continuous Treatment DiD

Uses actual **cbd_congestion_fee** as a continuous dose variable.
Estimates dose-response curve: how much does each additional \$1 in fee reduce trip volume?

**Model:** $Y_{it} = \alpha_i + \gamma_t + \beta_1 \text{Post}_t \cdot D_i + \beta_2 D_i^2 \cdot \text{Post}_t + \varepsilon_{it}$

In [ ]:
from src.causal.run_continuous_did import run_continuous_did, run_dose_response_by_bin
from src.evaluation.plots import plot_dose_response, plot_dose_bins

if 'avg_cbd_fee' in panel.columns:
    cont = run_continuous_did(panel, outcome='log_trip_count', quadratic=True)
    print(f"β_linear = {cont['beta_linear']:.4f}  (per $1 in congestion fee)")
    print(f"β_quad   = {cont['beta_quad']:.4f}  (non-linearity)")
    print(f"p_linear = {cont['p_linear']:.4f}")

    fig = plot_dose_response(cont['dose_response'], cont['beta_linear'])
    plt.show()

    # Non-parametric version
    bins = run_dose_response_by_bin(panel, outcome='log_trip_count', n_bins=5)
    fig2 = plot_dose_bins(bins)
    plt.show()
else:
    print('avg_cbd_fee not in panel. Need post-2025 data with cbd_congestion_fee field.')

## L4 — Synthetic DiD

Constructs a **synthetic control** by reweighting control zones to match pre-trends,
then applies DiD on the balanced panel. Most robust to parallel-trends violations.

In [ ]:
from src.causal.run_sdid import run_sdid

sdid = run_sdid(panel, outcome='log_trip_count', n_bootstrap=100)

print(f"SDID τ̂  = {sdid['tau_hat']:.4f}")
print(f"SE      = {sdid['se']:.4f}")
print(f"95% CI  = [{sdid['ci_low']:.4f}, {sdid['ci_high']:.4f}]")

# Compare L1 vs L4
l1_ate = l1_results.get('log_trip_count', {}).get('ate', None)
if l1_ate:
    print(f"\nL1 TWFE ATT  = {l1_ate:.4f}")
    print(f"L4 SDID  τ̂  = {sdid['tau_hat']:.4f}")
    print(f"Difference   = {sdid['tau_hat'] - l1_ate:.4f}  (should be small if trends were parallel)")

## L5a — Double / Debiased ML

**Why DML?**  
Zone FEs don't absorb time-varying confounders (e.g., a zone had a stadium event,
or a road closure). DML uses ML to partial out the full covariate vector $X$
from both $Y$ and $D$, yielding an asymptotically unbiased ATE.

**Cross-fitting** eliminates regularisation bias from the ML step.

In [ ]:
from src.causal.run_dml import run_dml
from src.evaluation.plots import plot_dml_results

if feature_cols:
    outcomes_dml = ['log_trip_count', 'avg_distance', 'avg_duration', 'avg_fare']
    outcomes_dml = [o for o in outcomes_dml if o in panel.columns]

    dml_df = run_dml(
        panel, feature_cols, outcomes_dml,
        learner='random_forest', n_folds=5, n_reps=3
    )

    print("\n=== DML Results ===")
    print(dml_df[['outcome','theta','se','p_value','sig']].to_string(index=False))

    fig = plot_dml_results(dml_df)
    plt.show()
else:
    print('Feature matrix not built. Run step 4 of the pipeline.')

## L5b — Causal Forest: CATE Estimation

Estimates **heterogeneous treatment effects** $\tau(x)$ for each zone × day unit.

Key outputs:
1. Per-zone CATE map
2. Feature importance: what drives heterogeneity?
3. Policy tree: simple interpretable rule
4. Best Linear Projection: which covariates linearly predict τ̂(x)?
5. RATE curve: gain from targeting high-CATE zones

In [ ]:
from src.causal.run_causal_forest import run_causal_forest, cate_by_group
from src.evaluation.plots import plot_cate_diagnostics, plot_rate_curve

if feature_cols:
    cf = run_causal_forest(
        panel, feature_cols,
        outcome='log_trip_count',
        n_estimators=500,   # reduce for notebook speed; use 2000 in production
    )

    print(f"Mean CATE : {cf['mean_cate']:+.4f}")
    print(f"Std CATE  : {cf['std_cate']:.4f}")
    print(f"\nTop 5 CATE drivers:")
    print(cf['feature_importance'].head(5))

    print(f"\n=== Policy Tree (depth-2 rule) ===")
    print(cf['policy_tree'])

    fig = plot_cate_diagnostics(cf['cate_estimates']['cate'], cf['feature_importance'])
    plt.show()

    if cf.get('rate'):
        fig2 = plot_rate_curve(cf['rate']['toc_curve'], cf['rate']['rate_score'])
        plt.show()
else:
    print('Feature matrix not built. Run step 4 of the pipeline.')

## CATE Heterogeneity: By Borough and Peak Hours

In [ ]:
if feature_cols and 'cf' in dir():
    if 'borough' in panel.columns:
        boro_cate = cate_by_group(cf['cate_estimates'], 'borough', panel)
        print("CATE by Borough:")
        print(boro_cate[['borough','mean_cate','se_cate','ci_low','ci_high']].to_string(index=False))

    if 'is_weekend' in panel.columns:
        wknd_cate = cate_by_group(cf['cate_estimates'], 'is_weekend', panel)
        wknd_cate['label'] = wknd_cate['is_weekend'].map({0: 'Weekday', 1: 'Weekend'})
        print("\nCATE by Weekday/Weekend:")
        print(wknd_cate[['label','mean_cate','se_cate']].to_string(index=False))

## Methods Comparison Table

Side-by-side summary of all estimators on the primary outcome (log_trip_count).

In [ ]:
rows = []

if 'log_trip_count' in l1_results:
    r = l1_results['log_trip_count']
    rows.append({'Estimator': 'L1: TWFE DiD',
                 'ATT': r['ate'], 'SE': r['se'], 'p': r['p_value'],
                 'Notes': 'Baseline; no ML confound control'})

if 'cs' in dir() and cs.get('agg_att'):
    rows.append({'Estimator': 'L2: CS-DiD',
                 'ATT': cs['agg_att'], 'SE': np.nan, 'p': cs['pre_trend_pvalue'],
                 'Notes': 'Honest ATT; pre-trend test p shown'})

if 'sdid' in dir():
    rows.append({'Estimator': 'L4: SDID',
                 'ATT': sdid['tau_hat'], 'SE': sdid['se'], 'p': np.nan,
                 'Notes': 'Synthetic control weights'})

if 'dml_df' in dir():
    r = dml_df[dml_df['outcome'] == 'log_trip_count'].iloc[0]
    rows.append({'Estimator': 'L5a: DML',
                 'ATT': r['theta'], 'SE': r['se'], 'p': r['p_value'],
                 'Notes': 'ML-debiased; RF nuisance learner'})

if rows:
    comp = pd.DataFrame(rows)
    comp['Sig'] = comp['p'].apply(
        lambda p: '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else '—'))
        if not pd.isna(p) else '—')
    comp['Pct Change'] = comp['ATT'].apply(
        lambda x: f'{(np.exp(x)-1)*100:+.1f}%' if not pd.isna(x) else '—')
    print(comp[['Estimator','ATT','SE','Sig','Pct Change','Notes']].to_string(index=False))

## Robustness Checks

In [ ]:
from src.causal.robustness import run_all_robustness
from src.evaluation.plots import plot_robustness

print('Running robustness checks (this may take a few minutes)...')
rob = run_all_robustness(panel, feature_cols, outcome='log_trip_count')

main_ate = l1_results.get('log_trip_count', {}).get('ate', 0)
main_se  = l1_results.get('log_trip_count', {}).get('se', 0)

fig = plot_robustness(rob, main_estimate=main_ate, main_se=main_se)
plt.show()

print(f'\n{len(rob)} robustness checks completed.')
print(f'Main estimate: {main_ate:.4f} ± {main_se:.4f}')
sig_share = (rob['p'] < 0.05).mean() * 100
print(f'{sig_share:.0f}% of robustness checks also significant at p<0.05')